In [3]:
from pathlib import Path
import pandas as pd
from transformers import AutoTokenizer


def filter_single_token_datasets(
    model_id: str = "sarvamai/sarvam-2b-v0.5",
    input_dir: str = "translation/data/method1/clean/final/",
    output_dir: str = "translation/data/method1/clean/single_token/",
    lang_codes: list[str] | None = None,
):
    """Filters language CSV datasets to keep only rows where {lang_code}_word

    encodes into exactly one token. Saves 2 columns per file: ['en_word',
    '{lang_code}_word'].
    """
    if lang_codes is None:
        lang_codes = ["ben", "mar", "tel", "tam", "hi", "en"]

    input_path = Path(input_dir)
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)

    print(f"Loading tokenizer: {model_id}...")
    tokenizer = AutoTokenizer.from_pretrained(model_id)

    for lang_code in lang_codes:
        csv_file = input_path / f"{lang_code}.csv"
        if not csv_file.exists():
            print(f"File not found: {csv_file}, skipping.")
            continue

        df = pd.read_csv(csv_file)
        target_col = f"{lang_code}_word"

        if "en_word" not in df.columns or target_col not in df.columns:
            print(
                f"Required columns missing in {csv_file}. Columns found: {df.columns.tolist()}"
            )
            continue

        # Keep valid string entries and check single-token condition
        def is_single_token(word):
            if not isinstance(word, str) or not word.strip():
                return False
            ids = tokenizer.encode(word.strip(), add_special_tokens=False)
            return len(ids) == 1

        mask = df[target_col].apply(is_single_token)
        filtered_df = df.loc[mask, ["en_word", target_col]].reset_index(
            drop=True
        )

        out_file = output_path / f"{lang_code}.csv"
        filtered_df.to_csv(out_file, index=False)

        print(
            f"[{lang_code:>3}] Retained {len(filtered_df):>3} / {len(df):>3} single-token rows -> Saved to {out_file}"
        )


if __name__ == "__main__":
    filter_single_token_datasets()

Loading tokenizer: sarvamai/sarvam-2b-v0.5...
[ben] Retained  93 / 132 single-token rows -> Saved to translation/data/method1/clean/single_token/ben.csv
[mar] Retained  98 / 132 single-token rows -> Saved to translation/data/method1/clean/single_token/mar.csv
[tel] Retained  81 / 132 single-token rows -> Saved to translation/data/method1/clean/single_token/tel.csv
[tam] Retained  70 / 132 single-token rows -> Saved to translation/data/method1/clean/single_token/tam.csv
[ hi] Retained 117 / 132 single-token rows -> Saved to translation/data/method1/clean/single_token/hi.csv
[ en] Retained 128 / 132 single-token rows -> Saved to translation/data/method1/clean/single_token/en.csv
